# Finetuning y Entrenamiento de Piper TTS (VITS) en Español Argentino

Este notebook contiene el pipeline integral para:
1. Verificar GPU NVIDIA (RTX 4060) y entorno PyTorch.
2. Procesar y estandarizar el dataset rioplatense (22.05 kHz mono 16-bit PCM).
3. Entrenar y finetunear el modelo Piper VITS con aceleración CUDA.
4. Exportar el modelo a ONNX y reproducir el audio generado interactivamente.

## 1. Verificación de Entorno y GPU CUDA

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Preparación y Resampleo del Dataset Rioplatense

In [ ]:
!python prepare_dataset_piper.py

## 3. Entrenamiento y Finetuning con PyTorch y CUDA

In [ ]:
!python train_piper.py --dataset_dir data/piper_dataset --epochs 20 --batch_size 16 --export_onnx

## 4. Validación de Síntesis y Reproducción de Audio

In [ ]:
import wave, io
from piper import PiperVoice, SynthesisConfig
from IPython.display import Audio, display

voice = PiperVoice.load("../voices/piper_ar.onnx", config_path="../voices/piper_ar.onnx.json")
text = "Che, ¿cómo andás? Te confirmo que el finetuning del modelo en español argentino se completó exitosamente."

buffer = io.BytesIO()
with wave.open(buffer, "wb") as wav_file:
    voice.synthesize_wav(text, wav_file, syn_config=SynthesisConfig(length_scale=1.0, noise_scale=0.667))

buffer.seek(0)
display(Audio(buffer.read(), rate=22050, autoplay=True))
print("¡Audio sintetizado con éxito!")